# Track A 06: Llama Power-Cap Robustness Analysis

This CPU-only derived analysis verifies two immutable evidence bundles before reading any result: the completed Llama v5 systems experiment and the completed Notebook 5 C1-W control. It then reintegrates all 100 raw NVML traces under three regimes: unchanged raw power, samples capped at 82 W, and samples capped at 70 W.

Raw joules remain the primary result. Capped values are sensitivity analyses for isolated NVML readings above the nominal T4 power envelope; source traces and stored rows are never edited.

## Run Order

1. Use a normal Google Colab **CPU runtime**; no GPU is required.
2. Confirm `CSE407_llama_v5_final/` and `CSE407_llama_track_a_full_v1/` are in the root of My Drive.
3. Run Section 1 once, then select **Runtime > Restart session**. This restart is mandatory because NumPy and SciPy contain compiled binaries.
4. After restart, continue from Section 2 and run all remaining cells in order. Do not rerun Section 1 in the restarted session.
5. Accept outputs only if both 95-trace and 5-trace manifests pass byte-for-byte verification and every raw trace reintegrates to stored joules.

Output is isolated under `CSE407_llama_power_robustness_v1/`.

## 1. Install Analysis Dependencies

This cell installs a mutually compatible pinned CPU analysis stack. When it finishes, select **Runtime > Restart session**, then continue from Section 2. A restart is required even if Colab does not display its own restart warning.

In [ ]:
%pip install -q "numpy==2.5.1" "pandas==3.0.5" "scipy==1.18.0" "matplotlib==3.11.1"
print("Dependencies installed. Restart the session, then continue from Section 2.")

## 2. Imports, Embedded Manifests, Paths, and Fingerprint

Both accepted artifact manifests are compressed inside this notebook. The analysis therefore verifies every source file without writing a manifest into either immutable directory.

In [ ]:
import base64
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import zlib
from pathlib import Path
from textwrap import dedent

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import scipy
    import scipy.stats as stats
except (AttributeError, ImportError) as error:
    raise RuntimeError(
        "The pinned NumPy/SciPy stack is not active. Select Runtime > Restart "
        "session, then resume from Section 2 without rerunning Section 1."
    ) from error

SCHEMA_VERSION = "track-a-llama-power-robustness-v1-2026-08-03-1"
EXPECTED_FINGERPRINT = "91ebc63773e735e6"
V5_FINGERPRINT = "8d27af8c9ee94de3"
FULL_FINGERPRINT = "106a561c07c62937"
V5_MANIFEST_SHA256 = "a7087d7e2e24a929df0f62177bc6fc1e7be38cd1bf12d8ffaf9849f6c7677b19"
FULL_MANIFEST_SHA256 = "2a4c996f32916b36aa11ad3e18104e6378ed7ad7a54230d8f728b7c1f52ba9bc"
CAPS_W = (82.0, 70.0)
THRESHOLDS_W = (70.0, 82.0, 100.0)
PINNED_VERSIONS = {
    "numpy": "2.5.1",
    "pandas": "3.0.5",
    "scipy": "1.18.0",
    "matplotlib": "3.11.1",
}
CORE_VARIANTS = ("C0", "C1", "C2", "C3-Ref", "C4-Ref")
CONTRASTS = (
    ("C1", "C2"),
    ("C1", "C3-Ref"),
    ("C1", "C4-Ref"),
    ("C2", "C4-Ref"),
    ("C3-Ref", "C4-Ref"),
)
V5_MANIFEST_BYTES = zlib.decompress(base64.b64decode(
    "eNqtnNtuZMd1hu8N6B0IXZtKnQ/Jla0IiADbQWIlN0FA1HHUzgwpN3tGGhh693yLmqHkpHarm3sgWNYMm+Tq6qpV/2ntv332m5ubz9+U+8Mcj6e7x/bteFM+/8ebz9vjcCreluPpMEs73X58ye07/flvn75p/PDdOB7ejPvT3aHLt7x+Xd6UW/uFubX19p2/PY45juO+jQ/f8PFn3R0fHk7yDV/++St+x93T992983fzcF9ef3jx46mc3j7Kq77+4x//45vf/f4PX939y9d//uZf//3rL3/3h7uv/vPrf/7qT19+9eHV3z28PrT38uo/jXfjeDP64XTzcLx54A/fHw+ncTMPr8fjzdv7zlf/rpB/uuE1h/n+prwqh/vH083p28Pjzce3+8WHXyAvl2r+Jn+S8p4W6o5vfTw83Msv5g0/1X9rlAm3LJ3Vtx+WSr7/cP9qHL87Hu6f3nnqJpaZWh4juz7s8+sex6P8wA8rKj9KRau/0cZbr29L19Ubl59f/uahj9d38zsd7o7j3eFjLUqF6Xvr2mbbbOndhND5PHN2teRgjB85pNz/z88p3//1736Miaba2kcKsQYVYlDOxT5qLo76dZ61z27LeP4xp7fH+vDXt4Ut8fjw9tjG3eO3xfggPyxan4aJMbXUR8jRVdWi0aaMNGyvQynvDcVVM0yfqhRnUkzTWH4bLyzPv4RFfFOO7+/GD6O9PcliHR++l8/GJvX8mofvx/HudCyU0B7ePi169vLFH3/6PA/379i4D8f3v/hMZYs8v1pr//yuHk7l9V19fxryW3TU3mZ98W962nj85X/99A0ffpl8Zzl9KwvzxavD6fDq/uH4vI588fm3mZ//7ue1DEmn6YItulutrBndKbZK8rPM0RSfmAuz8MdZsml1RLaAMlVnX6rtJX3+08/88bdbNf3iXT3+w5fqrp1+0Mq4u/r6of2P+uIvj2yQ/1+sD8Euyy3BhmmdGYFPOzqtFZ9/CXUGHVwoSc9oZ87RqhwofDRbdfGjGT2bm3lfuXqzXOOWq9tbKCXaUlId2pfcazGzZlY512FU6dNka7zpzrQ5ZlMxqlZtK7ak7nvdV645U25YlevmHDakWafWpmdXxki+xuZzDMn64YeTM+YyG8TwrnpRZfaY0oiz8h37yrVnNoNarm7R3sduEyfemKpcrNYW7/OcKvrgVZ55aNem09ISlA+GNzN0UdFYb8y+ct326tq4KnfOOkJVUp3Plq5KhaEH21yOkf+llKwuKWXdsx7KDfpVi3PmEG3yTr2kXKNc+pWjxq/2y9XlyExdrSkh+BTZyaHoFIu3s/D5By4d3SYLW7uhxbJ7tY656dRz9/5lR+3ncvWZcperm3z2ih6VrPJ2tKBU5BiF1qRVedta62Uow3+r5pRtegY3K1tbN143w75yN49a9Gq5d0PrOTeab2XdqrazRfnATaaXxVqKdb0p10xvVZnSrLchexbaG12zG2VfuZtHLQa37LtjWs45nVZTosnVJgeemNWwNdjMo2g2Ni2CmjurbK2aWgU3WjeDta/7ynVnVnfZd712XlUD0ODo6D61sdxWIJ6pI7edSq7wj1PTtJSKbtYK+MipxZCtUy9qZGCi8CtHTbsQ86reyIJx0fbi9MzGDs1ahup0DjQN37MfalQut8aVq6nZxGGGD7q6wGr3sq9evV1vMsv1dTVll50Kw/WYa+K41Wo8+zanyklTwwH4DN/eK5fydIoG1lTutGUdw8713Txs2qu0vNi6tdl7G1mq2FpqllbMzVVUbbNn2kRWM1ZVqta2TFuDn64CFekPc+i962u36+WkrOotLo2uejKhm9Sc52JusjG9L1E+/+rAtqoXoxq3m/fcfDXOFGehHce+c33d9n7Ibtl8jbMqdjP7AAilGUAQMdMfhmmKu8M4NQKAYqQ+OYIVcKdD7BSe8gT/XNse9KUo0sa0bA+WYoECqvDxq5EbKBwc4WRlB70j6Vxy8XAcqd/1HiaNTrsamnXOhn3lbh634KJblatD7qMMzn/xCeztvK5TSgeyGwtG5++qlj7HbRfZyiYnbkKdUvPwI7uv3O2rTVm93LwtsglTmhqIozlD3GGg3yTX7PC+e5oYGJ3PgHJDMNpNlWtpPflu/d7NsI0ibVr23kqrgpbazn3TU6GhwnHbgE32EOgE0XHGPJvDepcaf5np1IWbLjvrfFH7yt08a4HjvgYO1ljorOt1KlCMkHKVepfeVZ0dkQXtWhdQOlyOuySUDmjgP6bqsb1oM1yAIkPQfg0cTLfKZG4plIdsetI1xepb6bF755JqBvDIhTF0Gz6PmADqcjXzYSg795W7fdTC+mKjqCwgx7uaEy8qk6UDLXLbZY4c32UU7QDABgI2RrOJY6uThYdz0tf2lXvuqC3vtTShZpXaDPtRcY6gjp29kGFuYDQLalMNAEYTgO/AfUoQQjxcoCmPsnMz2DPlLjfDNK4LvUTNgwpnekO1uXaldAc2hiAqVsioDUKZe1QZpK5iolDqntbvK9ed2bvLzZB66I5fb6tQSu2At3QtqFtnW7QSOmCyjdorpN1zj3Topq1qJCQHULt+SbmXoEhr9PIWLhz2XgIkDWwIssitOYBYQ7EZlX7mE2i9gXT4+OtwzkHW4KLc3VwkZZZ99W6jSOvy8qLgZm3QdnQnSFtMXc0e2JUjNASFznuoDfURbAEnjqOo7lIEPnIHT81L8756zbn1Xd7DzSUQC31ZGikCTcuaxTQZFgKWMAZ4m1FHRnDGAXlBd6z3EP7eHSBu5/puo0gLO1z2XmjjTDN32oLh2kUp81pwrgZQwiAU5J7rzmroUEeWEl0V+Osh8QWg6fbV667eDyAXjhSkB3LTNLoyMmPg3yAzNGPutInw7BNNrSJDIgEKxVPVW6tbsu1aLdLc/h79NVx66rQOa67ZEMDtUy02xBmRHAb3XfEwDW7gMaPubJooeL2oGVjvGAVZhkHdunyKqrfPHp/tshUH6gGkO2/QH6JCQPVSnBl8gwNlZiims+wKhL3IxlEZwQRAySuSNlN/iqq3TyD8dwl+0Pkp0AAOnGjR4FzchD7y7BMCHWm+0beI2soFhA5koJtJJSX2SFbsn09RtT2zQ9bnMOugLRdZLTpg17BXa09dRHdnaA4qqjR7ajOAMTT/tM5b4+0IIlXWfpKq3bmql6fRRRHU4ZkoqjXTY9DVDEYAAB54jz5JmRqPIOaiHaeV/861VajKDM05c33VXpuLDyMSyPIKRCsDUoLLkHI8to/R6CiRDYHQxkYoEY5RWGzgnUvIEUPT7FVEeEFVu1qnWhV95izCn9crjWaGm8X6qtppIMhmKiXjZze0u47KysYwCRAdWoSJRA/7AzdrsAYc9hMUfeYouvVKOxQ1JbuY+47WjAcZWugVKRB/caCuDIM8nE2JQvbRXaEkYnyU3ASkfIqit08ioH3ZqzlioGHF5g0jxj65/AanEQUbHwaWn3yrVD1sQke2JStYGCZde2om6mqDY1W0O7fSS5BvaXPIbBZPQEcX2cKjYnYZDqXKGHf4usOhCuDNmmAn7tgAnXC3awTlkK/e0xeKK8GlJUqiF6NkI6AhZDrsQ/oDHQ5IRE0wasuGGexsW+nM0YM86CTJIXaW6hNm2b5yty06r5eHDwCH04FcguDnR09saRRMejCLiZ3O23DF0uSA1ViOCEOoh8VjlaIgulLMvnLNmXLXyuDIKnN1P/mJtAtaRQEcu2LgTHOUOKgOta0EJEJ2R0HYwqxPSnyDef0NeKm4ktdCptKyEwBHHC2YfggBzoFJayJ9i5/LmipEQvYS5kFCMKZxD4Mag0+TW965utsWXfBLVKS5GDjy3GO8gEOHsxzSICghqgrKH8TDuKh8o18APVUF+heDwYR/UKn4JeVeYtGldbncuFgDXuPJIhVP3FtYP+oEm3aYSZPDiwuxEILwMGhgJ+0BL1yxz7kSrzaRzKUW3Ua5LQs+Q/9NPtgE4bSYs92L4OI60hA+L3sX2o/3MZA14NoNRhhwozE/gtlXrjlD//NadvUZc0vwDF2Kq6FDmlhsXz30GilTkbxRFnaNnRwRMIieiI/A3rUF9XVfuZtHjbTN8kLmdpB+hFDSLLsWOwYShfpWLYhNgeiHDvTmyRVskImibdjPJHCmRasniLOv3G2LDitrCYrZufQGg5VBcbiIsaAIIXBzIYj9JZoECIK7OURaRtANqTsgHLLaoCX/knIvQpZkHZabl41a8L0GhB4fQ8gbiQ3RBlnvyLchv6FbwEwNCm1yI6LDox6KVCTi/b56z4FKv5YG2bC4Nd1zz0piww6FKVAyRhGhCW5lCiQxRlfLJCRiA+dXEcIhIcb2FPfVa86B4CVsoC+UmRV+4cTKmJjexHZGyuAEfNuIRQeLa6BkUE+BJCFc+GGIJKHKkJro++o9R+rCkh6xWPhatAGWF/CuAWlQvIkFSl/D15uNxBSNkOQUx5E3VNGL5NYD3rNRzL56z9G5tUWnhhc+D6HsrCLCH1S/TeOB7krxZzFH8WxMQnKjSdP3EL7RuAAYKDH9WpvD3n7zbxefOeyMdQdGhpq4F8OT2kEAbDQ4FKwpcUG5iDlkkowAVXIHd5GTg3jOCV6q4NNtf836XM3LvhYnH3mrgErY81BgXy43Q0pN5GFSdOwFVeD2taAj8kLh0RilKXMuAzR0f83bZw/k5dfC24C04SAPnAEINPQBPk9Xo4EFoHCnAxeylAUxCG3LS5TNoQEUTDvyn25/zfbcOi+pHJBc8UVAJJdIcx2qpiTrOcRCtNC2AH7sqEGIssiJtGzdDbKbAwhLMmR/ze7MOqu1OkH2jAU21CKqSpeCNQqLw2sCwykHESUnwU4WAzJgMGLfoFKgg3MO3Uv2s734DGa7tkcTOAyCJn5z0YOyMEVhcBhgRBW5uHmvqHGDlAoxXtwncD4MuwYaC1Hhub9mfa7mdTAwY9uCy5xUCJknwFq1QPcxuQ6xpWecqCc4u1hhCAFFYq1IoUjLiLnK7q/ZnKt5nV9Ddm10C1aYYGvG+cAcR1nGFHsKIHRO3ZSUILi4sM25S4BtJHY7Mj7e7v6a7bmalzkKWh2LScPlip4+SCicpCJYIoq9BNcL9DQy3xWzjCCpeOtlKnYFPb2S491fsztXs17DZNwHa3FDgfVdBEyWMOPVZCAFGwb3dCQJBIYBiIY0Jz24D5UkNgEb11I8e6GaEoE4yzYXIEq0LUAE/mOWUgdSVEFhDR0Ji7uvo4nHrJFVMncNIQBFTsXw2YDw3b5ytyleXHMmGBDeOPFFMxApUL2JWiH4EE9CDILgofg8iSvFoWF2CB6WNX0b+b5pQm47V3fbP3drF2SiTTlWk+kA4l4xSkCQVCaxZkwqWhyYSXSsxhvCjCKElTRKd0MQbNDY6xvyhWrK1mYg30ynwtWHo4yaJBCIey77FZLXKvFMph6gqS7zB6LkyPVPKiYxC9Lkuu4rd/OsZbWmTDFigBJNwO1gXa3wZ4gQgKETeTXsUKMkrUTyEhiXaL/AtiFGX7DAivKi1b1ATSFtvS4XWxEkIck6osHkaxA44XETfzxzpzmCamgA0FP8pID0AjpC7WbuhisapB/2lbt51FBD1l6YnHM5hyjXGkOM1HiXXCtRqop+gmOKrEayYnAJMi4D0UZaEducA4hRVveVa86Uu8RrQGD0BsRqPmqySkomCoYE1CDSTBN4cQqAOpZYtCNFqAPyBKEwRg1IgdAo9pVrry3XOxIqYHmcO9YLgRWsRsoVXUJ1Rl+AyN5JCIi9TS4sSsjRIBOhdzoOzPWw50I1hRzP+ibGOyA5AxO1uLoIUAkxRQtva2B0HA5JlEKPMu159J5lFAwRrisP9QKKvqTcS1AlKuna08AeQuZt2M9Z8hQVoheKtQGvnzkdkqGledzxBPVj7sySKWS8AxbVoNdk5ffVq8/Vu7woEmMENncaF8pDVETwqZUMNKY+uQnQMJy/g8Yiw0kobZHIR+PfYGQEgRr1vnrNuXrXUYoo8Xu+TFAbnwvA5aH1IDI0i5Blp7pAoIkRqfE0c8Q7iBA+7DCuwNzqvnrtmXp1WkeXiGUHhlzoYVGgDJ4ALj/JLxI1LLulGeMgOmKOTcbRNDEmDajEN6gME6Z99bpz65vXoQ/CVkSeqWFKdgIEzCyagHbGd7AHZLCvJ3owHBVSKvM9eB/siYLOxoG8sl53IYpMGzZMECGVBBsLx5U1gTPcY6KmTYhaAE1m0leJ7hBxOITrY8eA1kmSiuJq9b5yt1GkD2sLUcHSZuMsobOKpfkUvxY4g/oLwEFKIZKrOyq20gB5bE5PPyMNwuAmF+G+cjdPW7LZrdPvKCGoOywfvHIisSYAhFA5VCt0IBqtzBiQ/wA0IqmkQiAeHR6LFu316ubrLkSRtCe3ViFosCTTGB8hEaE0+AVwIFNQwHKyjoQuEU2ILiXuX7msi8TJQe2MfOEyzn3lujMYfYlzGDpjtJBgJdQxwsWg815yroxGwSwIh8HcNEFHuCgOOkMnKGwpE7vCMSJ4UF5S7iUoUq01S0ilEW3Pa457gkwEVPUqI8/0PrwWpmWwnJvksY0E9knSACQZribGX7jg9pW7fbMpt2HKkV9F3iEjiH/BzaplOk5mZplCKtkzV8tMCe4xs+LIUYBixnfIrUWaBPMKcefybsNINLFl42VikgEIsopcA6gkJMZp/6hj7ABizqXI0C8xJAetUIzX9kKMFOoGtyzNpJ3La8/shvVgDK6WkzZKJ2P0F5wDBmfrdhqy3L4QZVodWruwzSzsg1PJ7ARU2VWMpn3lbsPIsG4NbFA4JTOoBGUa80TMT5MPBIGxPxhkhAYhAjZQAiNHFnQMc0v0Z0Wn6zI59ZJyL5qbU+s8D/uQc0MlWBjYMaLxQeMJKhKBgKpZGAaFz8LqEpGBW/B7JQnNnElECtb76j2TeLZrwaxSa2aKU1RpVD2wbjCJZkC8pEMgnI+DqVpCMeI7KxlRQi4T+5sAQuJ47qt3G0ZCtzbkHM0UiY0RlUyct5kBOpJ7xveUXD4dAImHqUYr4548S4AsGGQYc5k5qnq1yO4uhZGEspbrK7MNAEgIehaM7sQjJJBPsIsMKwYB0VAUSCKMVuL7dN0uw6vMJND2SLju3A9n5uZQYNZ5NHYCIinEGPmU6BEbGB8IWapiBpGakaccsIuRTIhHke7i6RXEOoTTQePVr++Hj49C4YESRx6h8faNPO/ii/b4blEl/vo69h5gBYkgOfa2zLvkBnzxBG7B30QRUc3V4BVgmy5EXVnUEXIR+HNIkMjvVxZJMafHjRIJgsd1wjbxwA9uqKyUPI+Ex4AgMgtHwEoB0DJX0gXXki7h2AEjmeoUMIa4Szbpgj7wXOK4H8dX73m2yePb15tlMmuzzmjw64lPMyIJXkyWEQc2JWEjTGHSRnTYBvtq8jAO6G9l+ESCHI0voU03SSNeU+e7w/HhXp6ps52SXAf42K1SEUekyoBpx9XBxE7E8jH/xPkhqkFOX5Q9su04nRQdZSyVHAFwN11e5KG/ls/79XgzTmzLjTo3gIu41swIQRNljkExgs7YP8iLKb3OuZYngwQCsxk5BDtC9OfKAnPRwnhgYVecHZ60MzjxTw+IObc7w7opESfFb+BeShLvxdHBsY6SbUnM3hDqZI6ID5jqYOgkPBHxmNCArvFHycCMywt9w548PNS/DGp9N+6+K8dxeriTBrVRMQmxZbY34UEx5Q92HTLWgMUDOSD0JKoC47tkepLQSJg6j/1o5AiKRLTkQSFYJtds018v0a47J4+rIb/iOfeI3Sh06EyEBLCvM08lwVrAfcQXzoRxyLMUGdvsXhgGHzwsRrWrS3zL06eOJ57rdHr/U7nfIfItmih+wpKCQ0uRaeW5Hg7vSZOHhIMFGcqDkTOxRfNi0JznFeQMcOFxCmACpCOecMQMfrXlExR8/2q1af3GAzVAnwTcKtQ6oHXjs1M3lBxTh3kLenuVp+pwiRYm95gdkXFIHmjjafo8sqBdc754zNPrw+nXmmogjrIEgRxtD+ESoCITT6jFjggvzZUmSj5dInC91orkJdM5zM4BCuDobgaJx/YrCv1Y4HolJUNu1hQWvvIUEE08/QeZhQHTBmMgWV8Z8mdQjzw3djT3J5uEAB9TDIYRENILhCviBSbec4WP4/6RZ0C9O5z/0GVSxa8/dboRo1STh5CA9dsQG4QbktMvEWmUQh5IgfSF7AZgxage8tgHPF4YFw8eqPNlpX64Vc+0AQpe5xNIjuPls+lQUhDWhkxy4DsRMEXC8oS0LJo8ygY2r5NojiGaBbEVMIizV/rLCv64Zc9VHMKGz4SJ51hbgD3PJAMC8Jgi9PCKC0omi1Uk4a3IW0BoEcIZ7JenpEmWMEp0JLeXVfwB+529DfQ66Sb2Ec+WYzxernlJG0iGQuIqaoD7kV04/9y/hcuNUXMeFYdOjtoF96U1X4EHHt8/nsYbgdd9PMz5eG4LAz3XjwjjmT/kU/DvwQI8GO7pMVuM+STBqgO8gpKB1EXH4OvkICHLIDAGvlHHQWFaXV7uL55R1w+PPAJOEMIWJFyP7AJAWUxL8IsPHXFTIh8ImuiYnCrC/J4gBagBRoU8BIBALzIy5cG46cSc9JcX+44N28tThVuh+I2ZKiKvdCr2Jh4CojB7IUq3ml52MQErPFAaKmbklERxE0XAi8PI8Qtwr+cFlf/7789+8+Nnv/lf3ujw2g=="
))
FULL_MANIFEST_BYTES = zlib.decompress(base64.b64decode(
    "..."
))
assert hashlib.sha256(V5_MANIFEST_BYTES).hexdigest() == V5_MANIFEST_SHA256
assert hashlib.sha256(FULL_MANIFEST_BYTES).hexdigest() == FULL_MANIFEST_SHA256
EXPECTED_V5_MANIFEST = json.loads(V5_MANIFEST_BYTES.decode("utf-8"))
EXPECTED_FULL_MANIFEST = json.loads(FULL_MANIFEST_BYTES.decode("utf-8"))

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

FINGERPRINT_DATA = {
    "schema_version": SCHEMA_VERSION,
    "v5_fingerprint": V5_FINGERPRINT,
    "full_fingerprint": FULL_FINGERPRINT,
    "v5_manifest_sha256": V5_MANIFEST_SHA256,
    "full_manifest_sha256": FULL_MANIFEST_SHA256,
    "trace_counts": {"v5": 95, "c1w": 5},
    "caps_w": list(CAPS_W),
    "thresholds_w": list(THRESHOLDS_W),
    "pinned_versions": PINNED_VERSIONS,
    "integration": "phasewise numpy.trapezoid over elapsed_s and min(power_w, cap)",
    "core_variants": list(CORE_VARIANTS),
    "contrasts": [f"{left}-{right}" for left, right in CONTRASTS],
    "test_metrics": ["total_energy_j", "decode_energy_j", "decode_s", "decode_tok_s"],
    "correction": "Holm within the original 20-test core family for each regime",
}
EXPERIMENT_FINGERPRINT = canonical_hash(FINGERPRINT_DATA)[:16]
assert EXPERIMENT_FINGERPRINT == EXPECTED_FINGERPRINT
assert EXPECTED_V5_MANIFEST["facts"]["fingerprint"] == V5_FINGERPRINT
assert EXPECTED_FULL_MANIFEST["facts"]["fingerprint"] == FULL_FINGERPRINT
assert np.__version__ == PINNED_VERSIONS["numpy"]
assert pd.__version__ == PINNED_VERSIONS["pandas"]
assert scipy.__version__ == PINNED_VERSIONS["scipy"]
assert importlib.metadata.version("matplotlib") == PINNED_VERSIONS["matplotlib"]

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path("/content")
V5_ROOT = DRIVE_ROOT / "CSE407_llama_v5_final"
FULL_ROOT = DRIVE_ROOT / "CSE407_llama_track_a_full_v1"
SAVE_DIR = DRIVE_ROOT / "CSE407_llama_power_robustness_v1"
assert V5_ROOT.name == "CSE407_llama_v5_final"
assert FULL_ROOT.name == "CSE407_llama_track_a_full_v1"
assert SAVE_DIR.name == "CSE407_llama_power_robustness_v1"
assert SAVE_DIR not in {V5_ROOT, FULL_ROOT}
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def atomic_write_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

def atomic_write_json(path, value):
    atomic_write_text(path, json.dumps(value, indent=2, ensure_ascii=True) + "\n")

environment = {
    **FINGERPRINT_DATA,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": importlib.metadata.version("matplotlib"),
    "source_roots": [V5_ROOT.name, FULL_ROOT.name],
    "output_root": SAVE_DIR.name,
    "raw_joules_policy": "Raw trace integration is primary; capped values are sensitivity analyses.",
}
environment_path = SAVE_DIR / "power_robustness_environment.json"
if environment_path.exists():
    existing = json.loads(environment_path.read_text(encoding="utf-8"))
    assert existing["fingerprint"] == EXPERIMENT_FINGERPRINT
atomic_write_json(environment_path, environment)
print("Power robustness fingerprint:", EXPERIMENT_FINGERPRINT)
print("CPU analysis output:", SAVE_DIR)

## 3. Verify Both Immutable Evidence Bundles

The historical repository manifest contains an optional non-evidence `.gitignore`; its absence on Drive is accepted. Every scientific file must otherwise match its registered byte size and SHA-256 hash. Notebook 5 must match all 324 files exactly.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_manifest(root, manifest, optional_paths=()):
    assert root.is_dir(), f"Missing evidence root: {root}"
    expected = {row["path"]: row for row in manifest["files"]}
    observed_paths = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file()
    }
    required_paths = set(expected) - set(optional_paths)
    assert required_paths <= observed_paths, sorted(required_paths - observed_paths)
    assert observed_paths <= set(expected), sorted(observed_paths - set(expected))
    checked = []
    for relative_path in sorted(observed_paths):
        path = root / relative_path
        record = expected[relative_path]
        assert path.stat().st_size == record["bytes"], relative_path
        assert sha256_file(path) == record["sha256"], relative_path
        checked.append(relative_path)
    return {
        "artifact_root": root.name,
        "fingerprint": manifest["facts"]["fingerprint"],
        "expected_files": len(expected),
        "observed_files": len(checked),
        "optional_absent": sorted(set(optional_paths) - observed_paths),
        "verified": True,
    }

source_verification = {
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "schema_version": SCHEMA_VERSION,
    "v5": verify_manifest(V5_ROOT, EXPECTED_V5_MANIFEST, optional_paths={".gitignore"}),
    "notebook5": verify_manifest(FULL_ROOT, EXPECTED_FULL_MANIFEST),
}
assert source_verification["v5"]["observed_files"] in {114, 115}
assert source_verification["notebook5"]["observed_files"] == 324
atomic_write_json(SAVE_DIR / "source_verification.json", source_verification)
print(json.dumps(source_verification, indent=2))

## 4. Validate Registered Rows and Resolve Trace Basenames

Absolute Colab paths stored in CSV files are provenance, not portable local paths. This analysis preserves them and resolves each trace only by its registered basename inside the corresponding immutable `power_traces/` directory.

In [ ]:
def read_bool(series):
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.casefold()
    assert normalized.isin(["true", "false"]).all()
    return normalized == "true"

def safe_trace_name(variant, context, block):
    safe_variant = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(variant))
    return f"{safe_variant}_ctx{int(context)}_block{int(block)}.json"

import re

v5_core = pd.read_csv(V5_ROOT / "v5_final_energy_results.csv")
v5_sensitivity = pd.read_csv(V5_ROOT / "v5_final_sensitivity_energy_4096.csv")
c1w = pd.read_csv(FULL_ROOT / "track_a_c1w_energy_control.csv")
assert len(v5_core) == 75 and len(v5_sensitivity) == 20 and len(c1w) == 5
assert set(v5_core.fingerprint) == set(v5_sensitivity.fingerprint) == {V5_FINGERPRINT}
assert set(c1w.fingerprint) == {FULL_FINGERPRINT}
assert read_bool(v5_core.structural_pass).all()
assert read_bool(v5_sensitivity.structural_pass).all()
assert read_bool(c1w.structural_pass).all()
assert not v5_core.duplicated(["variant", "context", "block"]).any()
assert not v5_sensitivity.duplicated(["variant", "context", "block"]).any()
assert not c1w.duplicated(["variant", "context", "block"]).any()

final_integrity = json.loads((FULL_ROOT / "analysis/final_integrity.json").read_text())
assert final_integrity["passed"] and final_integrity["energy_rows"] == 5
V5_LABELS = {"C0": "C0", "C1": "C1", "C2": "C2", "C3": "C3-Ref", "C4": "C4-Ref"}
SENSITIVITY_LABELS = {
    "C2-B512": "C2-B512",
    "C2-B1536": "C2-B1536",
    "C3-TQ2": "C3-Ref-TQ2",
    "C3-TQ3": "C3-Ref-TQ3",
}

source_rows = []
for table_name, frame in (("v5_core", v5_core), ("v5_sensitivity", v5_sensitivity)):
    for row in frame.to_dict("records"):
        label_map = V5_LABELS if table_name == "v5_core" else SENSITIVITY_LABELS
        trace_name = safe_trace_name(row["variant"], row["context"], row["block"])
        assert Path(str(row["trace_file"])).name == trace_name
        source_rows.append({
            "source": "v5",
            "source_table": table_name,
            "source_fingerprint": V5_FINGERPRINT,
            "variant_original": row["variant"],
            "variant": label_map[row["variant"]],
            "config": row["config"],
            "context": int(row["context"]),
            "block": int(row["block"]),
            "trace_path": V5_ROOT / "power_traces" / trace_name,
            "trace_name": trace_name,
            "registered_trace_file": row["trace_file"],
            "average_budget": int(row["average_budget"]),
            "tq_bits": int(row["tq_bits"]),
            "stored_prefill_energy_j": float(row["prefill_energy_j"]),
            "stored_decode_energy_j": float(row["decode_energy_j"]),
            "stored_total_energy_j": float(row["total_energy_j"]),
            "decode_s": float(row["decode_s"]),
            "decode_tok_s": float(row["decode_tok_s"]),
        })
for row in c1w.to_dict("records"):
    trace_name = safe_trace_name(row["variant"], row["context"], row["block"])
    assert Path(str(row["trace_file"])).name == trace_name
    source_rows.append({
        "source": "notebook5",
        "source_table": "c1w_control",
        "source_fingerprint": FULL_FINGERPRINT,
        "variant_original": row["variant"],
        "variant": "C1-W",
        "config": "C1-W",
        "context": int(row["context"]),
        "block": int(row["block"]),
        "trace_path": FULL_ROOT / "power_traces" / trace_name,
        "trace_name": trace_name,
        "registered_trace_file": row["trace_file"],
        "average_budget": 1024,
        "tq_bits": 4,
        "stored_prefill_energy_j": float(row["prefill_energy_j"]),
        "stored_decode_energy_j": float(row["decode_energy_j"]),
        "stored_total_energy_j": float(row["total_energy_j"]),
        "decode_s": float(row["decode_s"]),
        "decode_tok_s": float(row["decode_tok_s"]),
    })
assert len(source_rows) == 100
assert len({(row["source"], row["trace_name"]) for row in source_rows}) == 100
assert all(row["trace_path"].is_file() for row in source_rows)
print("Registered rows and trace basenames validated:", len(source_rows))

## 5. Reintegrate All 100 Traces

Each prefill and decode phase is integrated independently with the trapezoidal rule. Raw power must reproduce all three stored energy fields. The capped regimes replace each sample `p` by `min(p, cap)` before integration; they never alter raw files.

In [ ]:
REGIMES = {"raw": None, "cap82": 82.0, "cap70": 70.0}

def integrate_phase(samples, cap=None):
    assert len(samples) >= 2
    times = np.asarray([row["elapsed_s"] for row in samples], dtype=float)
    powers = np.asarray([row["power_w"] for row in samples], dtype=float)
    assert np.isfinite(times).all() and np.isfinite(powers).all()
    assert np.all(np.diff(times) >= 0)
    if len(times) > 2:
        assert np.median(np.diff(times)) <= 0.25
    integrated_power = np.minimum(powers, cap) if cap is not None else powers
    return float(np.trapezoid(integrated_power, times)), times, powers

trace_rows = []
for source in source_rows:
    trace = json.loads(source["trace_path"].read_text(encoding="utf-8"))
    assert set(("prefill", "decode")) <= set(trace)
    if source["source"] == "notebook5":
        assert trace["fingerprint"] == FULL_FINGERPRINT
        assert trace["variant"] == "C1-W"
        assert trace["context"] == source["context"]
        assert trace["block"] == source["block"]
    record = {key: value for key, value in source.items() if key != "trace_path"}
    all_powers = []
    phase_arrays = {}
    for phase in ("prefill", "decode"):
        raw_energy, times, powers = integrate_phase(trace[phase])
        phase_arrays[phase] = (times, powers)
        all_powers.extend(powers.tolist())
        assert np.isclose(
            raw_energy, source[f"stored_{phase}_energy_j"], rtol=1e-9, atol=1e-6,
        ), (source["trace_name"], phase)
        record[f"{phase}_duration_trace_s"] = float(times[-1] - times[0])
        record[f"{phase}_samples"] = len(times)
    for regime, cap in REGIMES.items():
        energies = {}
        for phase, (times, powers) in phase_arrays.items():
            values = np.minimum(powers, cap) if cap is not None else powers
            energies[phase] = float(np.trapezoid(values, times))
            record[f"{phase}_energy_{regime}_j"] = energies[phase]
        record[f"total_energy_{regime}_j"] = energies["prefill"] + energies["decode"]
    assert np.isclose(
        record["total_energy_raw_j"], source["stored_total_energy_j"],
        rtol=1e-9, atol=1e-6,
    ), source["trace_name"]
    record["raw_reintegration_abs_error_j"] = abs(
        record["total_energy_raw_j"] - source["stored_total_energy_j"]
    )
    record["total_samples"] = len(all_powers)
    record["max_power_w"] = max(all_powers)
    for threshold in THRESHOLDS_W:
        record[f"samples_gt_{int(threshold)}w"] = sum(
            power > threshold for power in all_powers
        )
    record["cap82_reduction_j"] = record["total_energy_raw_j"] - record["total_energy_cap82_j"]
    record["cap70_reduction_j"] = record["total_energy_raw_j"] - record["total_energy_cap70_j"]
    trace_rows.append(record)

traces = pd.DataFrame(trace_rows)
assert len(traces) == 100
assert traces.raw_reintegration_abs_error_j.max() <= 1e-6
assert (traces.total_energy_raw_j >= traces.total_energy_cap82_j).all()
assert (traces.total_energy_cap82_j >= traces.total_energy_cap70_j).all()
traces.to_csv(SAVE_DIR / "trace_level_power_robustness.csv", index=False)
print("All traces reintegrated; maximum raw error:", traces.raw_reintegration_abs_error_j.max())
print("Maximum raw sample:", traces.max_power_w.max(), "W")
print("Samples >70/>82/>100 W:", *(int(traces[f"samples_gt_{x}w"].sum()) for x in (70, 82, 100)))

## 6. Condition Summaries and Ranking Stability

Core rankings are recomputed at every context and regime. Sensitivity rankings compare Pyramid budgets 512/1024/1536 and TQ-Ref numerical 2/3/4-bit settings at 4096. TQ-Ref uses one uint8 index per coordinate at every numerical bit width.

In [ ]:
summary_rows = []
grouping = ["source", "source_table", "variant", "context"]
for key, group in traces.groupby(grouping, sort=True):
    row = dict(zip(grouping, key))
    row.update({
        "n_blocks": len(group),
        "raw_mean_j": group.total_energy_raw_j.mean(),
        "raw_sd_j": group.total_energy_raw_j.std(ddof=1),
        "cap82_mean_j": group.total_energy_cap82_j.mean(),
        "cap82_sd_j": group.total_energy_cap82_j.std(ddof=1),
        "cap70_mean_j": group.total_energy_cap70_j.mean(),
        "cap70_sd_j": group.total_energy_cap70_j.std(ddof=1),
        "cap82_mean_reduction_j": group.cap82_reduction_j.mean(),
        "cap70_mean_reduction_j": group.cap70_reduction_j.mean(),
        "max_power_w": group.max_power_w.max(),
        "samples_gt_70w": int(group.samples_gt_70w.sum()),
        "samples_gt_82w": int(group.samples_gt_82w.sum()),
        "samples_gt_100w": int(group.samples_gt_100w.sum()),
    })
    summary_rows.append(row)
condition_summary = pd.DataFrame(summary_rows)
condition_summary.to_csv(SAVE_DIR / "condition_power_robustness_summary.csv", index=False)

ranking_rows = []
v5_core_traces = traces[traces.source_table == "v5_core"].copy()
for context in sorted(v5_core_traces.context.unique()):
    subset = v5_core_traces[v5_core_traces.context == context]
    assert set(subset.variant) == set(CORE_VARIANTS)
    for regime in REGIMES:
        metric = f"total_energy_{regime}_j"
        means = subset.groupby("variant")[metric].mean().sort_values()
        for rank, (variant, mean_energy) in enumerate(means.items(), start=1):
            ranking_rows.append({
                "analysis": "core",
                "context": int(context),
                "regime": regime,
                "rank": rank,
                "variant": variant,
                "mean_energy_j": mean_energy,
            })

sensitivity_sets = {
    "pyramid_budget": {
        "C2-B512": "C2-B512",
        "C2": "C2-B1024",
        "C2-B1536": "C2-B1536",
    },
    "tq_numerical_bits": {
        "C3-Ref-TQ2": "C3-Ref-TQ2",
        "C3-Ref-TQ3": "C3-Ref-TQ3",
        "C3-Ref": "C3-Ref-TQ4",
    },
}
sensitivity_pool = traces[(traces.context == 4096) & (
    traces.source_table.isin(["v5_core", "v5_sensitivity"])
)]
for analysis, label_map in sensitivity_sets.items():
    subset = sensitivity_pool[sensitivity_pool.variant.isin(label_map)].copy()
    assert set(subset.variant) == set(label_map) and len(subset) == 15
    subset["sensitivity_label"] = subset.variant.map(label_map)
    for regime in REGIMES:
        metric = f"total_energy_{regime}_j"
        means = subset.groupby("sensitivity_label")[metric].mean().sort_values()
        for rank, (variant, mean_energy) in enumerate(means.items(), start=1):
            ranking_rows.append({
                "analysis": analysis,
                "context": 4096,
                "regime": regime,
                "rank": rank,
                "variant": variant,
                "mean_energy_j": mean_energy,
            })
rankings = pd.DataFrame(ranking_rows)
rankings.to_csv(SAVE_DIR / "energy_rankings_by_regime.csv", index=False)

ranking_stability_rows = []
for key, group in rankings.groupby(["analysis", "context"], sort=True):
    orders = {
        regime: tuple(regime_rows.sort_values("rank").variant)
        for regime, regime_rows in group.groupby("regime")
    }
    ranking_stability_rows.append({
        "analysis": key[0],
        "context": key[1],
        "raw_lowest_to_highest": " < ".join(orders["raw"]),
        "cap82_lowest_to_highest": " < ".join(orders["cap82"]),
        "cap70_lowest_to_highest": " < ".join(orders["cap70"]),
        "cap82_matches_raw": orders["cap82"] == orders["raw"],
        "cap70_matches_raw": orders["cap70"] == orders["raw"],
    })
ranking_stability = pd.DataFrame(ranking_stability_rows)
ranking_stability.to_csv(SAVE_DIR / "ranking_stability.csv", index=False)
display(condition_summary.round(3))
display(ranking_stability)

## 7. Recompute the Original 20-Test Core Family

For each power regime, this cell recomputes the five declared 4096-token contrasts across total energy, decode energy, decode duration, and throughput. Holm correction is applied within the same 20-test family used by v5. Timing metrics are unchanged across power caps; energy metrics come from reintegrated traces.

In [ ]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    safe = np.nan_to_num(p_values, nan=1.0)
    order = np.argsort(safe)
    ranked = np.minimum(1.0, safe[order] * (len(safe) - np.arange(len(safe))))
    ranked = np.maximum.accumulate(ranked)
    adjusted = np.empty(len(safe))
    adjusted[order] = ranked
    return adjusted

def paired_test(frame, left, right, metric, regime, unit):
    pair = frame[frame.variant.isin([left, right])].pivot(
        index="block", columns="variant", values=metric,
    )[[left, right]].dropna()
    assert len(pair) == 5
    difference = pair[left] - pair[right]
    statistic, p_value = stats.ttest_rel(pair[left], pair[right])
    sd = float(difference.std(ddof=1))
    sem = float(stats.sem(difference))
    half = float(stats.t.ppf(0.975, len(difference) - 1) * sem) if sem else 0.0
    return {
        "regime": regime,
        "comparison": f"{left}-{right}",
        "metric": metric,
        "unit": unit,
        "n": len(difference),
        "mean_difference": float(difference.mean()),
        "ci95_low": float(difference.mean() - half),
        "ci95_high": float(difference.mean() + half),
        "paired_t": float(statistic),
        "p_unadjusted": float(p_value),
        "paired_d": float(difference.mean() / sd) if sd else np.nan,
    }

core_4096 = v5_core_traces[v5_core_traces.context == 4096].copy()
test_rows = []
for regime in REGIMES:
    metrics = (
        (f"total_energy_{regime}_j", "J"),
        (f"decode_energy_{regime}_j", "J"),
        ("decode_s", "s"),
        ("decode_tok_s", "tokens/s"),
    )
    regime_rows = [
        paired_test(core_4096, left, right, metric, regime, unit)
        for metric, unit in metrics
        for left, right in CONTRASTS
    ]
    adjusted = holm_adjust([row["p_unadjusted"] for row in regime_rows])
    for row, adjusted_p in zip(regime_rows, adjusted):
        row["p_holm_20"] = float(adjusted_p)
        row["significant_0_05"] = bool(adjusted_p < 0.05)
    test_rows.extend(regime_rows)
tests = pd.DataFrame(test_rows)
assert len(tests) == 60

frozen_tests = pd.read_csv(V5_ROOT / "v5_final_core_tests.csv")
assert len(frozen_tests) == 20
frozen_tests["comparison"] = frozen_tests.comparison.replace({
    "C1-C3": "C1-C3-Ref",
    "C1-C4": "C1-C4-Ref",
    "C2-C4": "C2-C4-Ref",
    "C3-C4": "C3-Ref-C4-Ref",
})
frozen_tests["metric"] = frozen_tests.metric.replace({
    "total_energy_j": "total_energy_raw_j",
    "decode_energy_j": "decode_energy_raw_j",
})
raw_tests = tests[tests.regime == "raw"].merge(
    frozen_tests,
    on=["comparison", "metric", "n"],
    validate="one_to_one",
    suffixes=("_recomputed", "_frozen"),
)
assert len(raw_tests) == 20
assert np.allclose(
    raw_tests.mean_difference, raw_tests.mean_difference_j,
    rtol=1e-10, atol=1e-8,
)
assert np.allclose(
    raw_tests.p_unadjusted_recomputed, raw_tests.p_unadjusted_frozen,
    rtol=1e-8, atol=1e-12,
)
assert np.allclose(
    raw_tests.p_holm_20, raw_tests.p_holm,
    rtol=1e-8, atol=1e-12,
)
raw_tests_match_frozen_v5 = True
tests.to_csv(SAVE_DIR / "core_4096_tests_by_regime.csv", index=False)

raw_significance = tests[tests.regime == "raw"].set_index(
    ["comparison", "metric"]
).significant_0_05
significance_rows = []
for regime in ("cap82", "cap70"):
    capped = tests[tests.regime == regime].set_index(["comparison", "metric"])
    for key, raw_value in raw_significance.items():
        capped_metric = key[1].replace("raw", regime)
        capped_key = (key[0], capped_metric)
        capped_value = bool(capped.loc[capped_key, "significant_0_05"])
        significance_rows.append({
            "regime": regime,
            "comparison": key[0],
            "raw_metric": key[1],
            "capped_metric": capped_metric,
            "raw_significant": bool(raw_value),
            "capped_significant": capped_value,
            "significance_unchanged": bool(raw_value) == capped_value,
        })
significance_stability = pd.DataFrame(significance_rows)
significance_stability.to_csv(SAVE_DIR / "significance_stability.csv", index=False)
display(tests.round(5))
display(significance_stability)

## 8. C1-W Descriptive Control and Power Anomalies

Historical C1 and Notebook 5 C1-W were collected in different sessions and used different 4096-token prompts. Their difference is reported descriptively only; no paired, causal, or isolated wrapper-overhead test is permitted. Every sample above 70 W, 82 W, and 100 W remains in raw evidence and is summarized rather than deleted.

In [ ]:
control_rows = []
control = traces[(traces.context == 4096) & traces.variant.isin(["C1", "C1-W"])]
assert control.groupby("variant").size().to_dict() == {"C1": 5, "C1-W": 5}
for regime in REGIMES:
    metric = f"total_energy_{regime}_j"
    grouped = control.groupby("variant")[metric].agg(["mean", "std"])
    difference = float(grouped.loc["C1-W", "mean"] - grouped.loc["C1", "mean"])
    control_rows.append({
        "regime": regime,
        "historical_c1_mean_j": grouped.loc["C1", "mean"],
        "historical_c1_sd_j": grouped.loc["C1", "std"],
        "c1w_mean_j": grouped.loc["C1-W", "mean"],
        "c1w_sd_j": grouped.loc["C1-W", "std"],
        "c1w_minus_c1_j": difference,
        "c1w_minus_c1_percent": 100 * difference / grouped.loc["C1", "mean"],
        "interpretation": "descriptive only; session and 4096-token prompt/workload differ; not paired, causal, or isolated wrapper overhead",
    })
wrapper_control = pd.DataFrame(control_rows)
wrapper_control.to_csv(SAVE_DIR / "c1w_vs_historical_c1_descriptive.csv", index=False)

anomalies = traces[
    (traces.samples_gt_70w > 0) | (traces.samples_gt_82w > 0) | (traces.samples_gt_100w > 0)
][[
    "source", "source_table", "variant", "context", "block", "trace_name",
    "max_power_w", "total_samples", "samples_gt_70w", "samples_gt_82w",
    "samples_gt_100w", "cap82_reduction_j", "cap70_reduction_j",
]].sort_values("max_power_w", ascending=False)
anomalies.to_csv(SAVE_DIR / "power_anomaly_inventory.csv", index=False)
display(wrapper_control.round(4))
display(anomalies.head(25).round(4))

## 9. Robustness Figures

Error bars are sample SD across five measured blocks. These figures compare integration regimes for the same immutable traces; they do not replace raw-energy figures.

In [ ]:
core_summary_4096 = condition_summary[
    (condition_summary.source_table == "v5_core")
    & (condition_summary.context == 4096)
].set_index("variant").loc[list(CORE_VARIANTS)]
x = np.arange(len(CORE_VARIANTS))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5.5))
for offset, regime, label, color in (
    (-width, "raw", "Raw", "#24445c"),
    (0.0, "cap82", "82 W cap", "#d77a32"),
    (width, "cap70", "70 W cap", "#5b8c5a"),
):
    ax.bar(
        x + offset,
        core_summary_4096[f"{regime}_mean_j"],
        width,
        yerr=core_summary_4096[f"{regime}_sd_j"],
        capsize=3,
        label=label,
        color=color,
    )
ax.set(
    title="Llama v5 4096-token energy under power-cap sensitivity",
    xlabel="Configuration",
    ylabel="Total GPU energy (J; mean +/- SD, n=5)",
    xticks=x,
    xticklabels=CORE_VARIANTS,
)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(SAVE_DIR / "core_4096_power_cap_robustness.png", dpi=180, bbox_inches="tight")
fig.savefig(SAVE_DIR / "core_4096_power_cap_robustness.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
plotted = condition_summary.sort_values("max_power_w", ascending=False).head(20)
ax.barh(plotted.variant + " @" + plotted.context.astype(str), plotted.max_power_w, color="#8f3d3d")
ax.axvline(70, color="#5b8c5a", linestyle="--", label="70 W")
ax.axvline(82, color="#d77a32", linestyle="--", label="82 W")
ax.set(title="Largest raw NVML samples by condition", xlabel="Maximum sample (W)")
ax.invert_yaxis()
ax.legend()
fig.tight_layout()
fig.savefig(SAVE_DIR / "maximum_power_samples.png", dpi=180, bbox_inches="tight")
plt.show()

## 10. Final Integrity and Reproducible Report

Acceptance requires 100 verified traces, raw reintegration within tolerance, complete ranking and test tables, and explicit disclosure of any ranking or significance changes. The final JSON is the machine-readable completion gate.

In [ ]:
ranking_all_stable_82 = bool(ranking_stability.cap82_matches_raw.all())
ranking_all_stable_70 = bool(ranking_stability.cap70_matches_raw.all())
significance_all_stable = bool(significance_stability.significance_unchanged.all())
max_row = traces.loc[traces.max_power_w.idxmax()]
required_outputs = [
    "source_verification.json",
    "trace_level_power_robustness.csv",
    "condition_power_robustness_summary.csv",
    "energy_rankings_by_regime.csv",
    "ranking_stability.csv",
    "core_4096_tests_by_regime.csv",
    "significance_stability.csv",
    "c1w_vs_historical_c1_descriptive.csv",
    "power_anomaly_inventory.csv",
    "core_4096_power_cap_robustness.png",
    "core_4096_power_cap_robustness.pdf",
    "maximum_power_samples.png",
]
assert len(condition_summary) == 20
assert len(rankings) == 63
assert len(ranking_stability) == 5
assert len(tests) == 60
assert len(significance_stability) == 40
assert len(wrapper_control) == 3
assert raw_tests_match_frozen_v5
assert all((SAVE_DIR / name).is_file() for name in required_outputs)
final_integrity = {
    "passed": True,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "schema_version": SCHEMA_VERSION,
    "source_manifests_verified": bool(
        source_verification["v5"]["verified"]
        and source_verification["notebook5"]["verified"]
    ),
    "v5_observed_files": source_verification["v5"]["observed_files"],
    "notebook5_observed_files": source_verification["notebook5"]["observed_files"],
    "trace_count": len(traces),
    "v5_trace_count": int((traces.source == "v5").sum()),
    "c1w_trace_count": int((traces.source == "notebook5").sum()),
    "raw_reintegration_max_abs_error_j": float(traces.raw_reintegration_abs_error_j.max()),
    "maximum_raw_power_w": float(max_row.max_power_w),
    "maximum_raw_power_trace": max_row.trace_name,
    "samples_gt_70w": int(traces.samples_gt_70w.sum()),
    "samples_gt_82w": int(traces.samples_gt_82w.sum()),
    "samples_gt_100w": int(traces.samples_gt_100w.sum()),
    "core_and_sensitivity_rankings_stable_at_82w": ranking_all_stable_82,
    "core_and_sensitivity_rankings_stable_at_70w": ranking_all_stable_70,
    "core_significance_decisions_stable": significance_all_stable,
    "raw_tests_match_frozen_v5": raw_tests_match_frozen_v5,
    "condition_summary_rows": len(condition_summary),
    "ranking_rows": len(rankings),
    "ranking_stability_rows": len(ranking_stability),
    "core_test_rows": len(tests),
    "significance_stability_rows": len(significance_stability),
    "raw_joules_remain_primary": True,
    "c1w_comparison_policy": "descriptive only; session and prompt/workload differ; no paired or causal test",
}
assert final_integrity["trace_count"] == 100
assert final_integrity["v5_trace_count"] == 95
assert final_integrity["c1w_trace_count"] == 5
assert final_integrity["raw_reintegration_max_abs_error_j"] <= 1e-6
assert final_integrity["source_manifests_verified"]
assert final_integrity["raw_tests_match_frozen_v5"]
atomic_write_json(SAVE_DIR / "final_integrity.json", final_integrity)

report = f"""# Llama Power Robustness Report

- Fingerprint: `{EXPERIMENT_FINGERPRINT}`
- Verified traces: {len(traces)} (95 historical v5 + 5 C1-W)
- Maximum raw sample: {final_integrity['maximum_raw_power_w']:.3f} W in `{final_integrity['maximum_raw_power_trace']}`
- Samples above 70/82/100 W: {final_integrity['samples_gt_70w']} / {final_integrity['samples_gt_82w']} / {final_integrity['samples_gt_100w']}
- Rankings stable at 82 W: {ranking_all_stable_82}
- Rankings stable at 70 W: {ranking_all_stable_70}
- Core significance decisions stable: {significance_all_stable}

Raw joules remain primary. Capped integrations are sensitivity analyses only.
Historical C1 versus C1-W is descriptive because both session and prompt/workload differ.
"""
atomic_write_text(SAVE_DIR / "power_robustness_report.md", dedent(report).strip() + "\n")
print(json.dumps(final_integrity, indent=2))

## Acceptance Gate

Accept Notebook 06 only when `final_integrity.json` reports `passed: true`, 100 traces, both manifests verified, raw reintegration error at or below `1e-6 J`, and explicit ranking/significance stability outcomes. A `false` stability outcome is a scientific result to report, not a reason to alter raw evidence.